# Classification de texte avec des RNN

Dans ces travaux pratiques, nous allons retrouver l'auteur d'une phrase avec des réseaux récurrents, écrits en PyTorch et entraînés avec Lightning. Le corpus : les phrases de 2 œuvres de chacun de 9 auteurs anglophones, de 1821 à 1913 (Brontë, Melville, Wilde…).

1. Du texte aux séquences d'indices.
2. Un classifieur récurrent qui ignore le *padding*.
3. L'entraînement d'un GRU bidirectionnel.
4. GRU simple ou bidirectionnel ?
5. Des embeddings GloVe pré-entraînés et figés.
6. Un modèle plus profond.
7. Les auteurs que le modèle confond.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Chaque entraînement y prend quelques minutes, contre un quart d'heure à plus d'une heure sur CPU.

## Installation et téléchargements

PyTorch Lightning et torchmetrics, le corpus, le modèle de spaCy pour l'anglais et les vecteurs GloVe de la partie 5 (un fichier de 822 Mo) : comptez une à deux minutes.

In [ ]:
!pip install -q lightning torchmetrics
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Import des bibliothèques

In [ ]:
import collections
import pathlib
from collections.abc import Iterable, Iterator

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import torch
import torchmetrics
import tqdm.auto
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, TensorDataset, random_split

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Prétraitements

La cellule suivante prépare les données :

- chaque phrase, découpée par spaCy, est un exemple, et son étiquette est son auteur, de 0 à 8 ;
- toute entité nommée (un mot ou un groupe de mots) est remplacée par son type, première lettre triplée pour en faire un mot nouveau (`PPPERSON`, `DDDATE`…) : le modèle doit reconnaître un style, pas les noms des personnages ;
- les dossiers du corpus portent des noms à coquilles (`Melwille`, `bronteCharlote`) : `AUTHORS` donne le nom de chaque auteur ;
- une phrase sans aucun mot (un crochet « [ » isolé par spaCy, de la ponctuation seule) est écartée : vides une fois découpées en mots, ces quelque 1 800 phrases auraient une longueur nulle, que refuse `pack_padded_sequence` (partie 2) ;
- 70 % des phrases servent à l'entraînement (`X_train_raw`, `y_train`), 30 % au test (`X_test_raw`, `y_test`), toujours les mêmes.

Elle prend environ deux minutes.

In [ ]:
# Dossier du corpus → nom de l'auteur
AUTHORS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  # <auteur>/<œuvre>/<chapitre>.txt
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.auto.tqdm(zip(paths, nlp.pipe(contents)), total=len(paths)):
    author = AUTHORS[path.parent.parent.name]
    for sentence in doc.sents:
      text = " ".join(replace_ners(sentence))
      # Une phrase sans lettre ni chiffre serait vide une fois découpée en mots
      if any(char.isalnum() for char in text):
        authors.append(author)
        texts.append(text)
  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))
label_encoder = sklearn.preprocessing.LabelEncoder()
y = torch.tensor(label_encoder.fit_transform(authors))
print(f"{len(texts)} phrases ; étiquettes 0 à 8 : {', '.join(label_encoder.classes_)}")

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  texts, y, test_size=0.3, random_state=0
)
print(f"Entraînement : {len(X_train_raw)} phrases, test : {len(X_test_raw)}")
for text, label in zip(X_train_raw[:3], y_train[:3]):
  print(f"{label_encoder.classes_[label]} : {text}")

## 1. Du texte aux séquences d'indices

La classe `Tokenizer` de la cellule suivante numérote les mots d'un corpus, comme le vocabulaire `vocab` des slides :

- un mot est une suite de caractères entre deux espaces, une fois le texte mis en minuscules et la ponctuation retirée ;
- `fit_on_texts(texts)` construit le vocabulaire `vocabulary`, la liste des mots dans l'ordre de leurs indices : `<pad>` (indice 0, le *padding*), `<unk>` (indice 1, tout mot hors du vocabulaire), puis les `max_tokens - 2` mots les plus fréquents de `texts`, du plus au moins fréquent ;
- `texts_to_sequences(texts)` remplace chaque mot de chaque texte par son indice.

In [ ]:
class Tokenizer:
  """Map the most frequent words of a corpus to indices, from 2 up."""

  # Caractères remplacés par des espaces avant le découpage en mots
  filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

  def __init__(self, max_tokens: int) -> None:
    self.max_tokens = max_tokens
    self.table = str.maketrans(self.filters, " " * len(self.filters))
    self.vocabulary: list[str] = []
    self.word_index: dict[str, int] = {}

  def split(self, text: str) -> list[str]:
    return text.lower().translate(self.table).split()

  def fit_on_texts(self, texts: Iterable[str]) -> None:
    counts = collections.Counter(word for text in texts for word in self.split(text))
    # Indices 0 et 1 réservés au padding et aux mots inconnus
    most_common = [word for word, _ in counts.most_common(self.max_tokens - 2)]
    self.vocabulary = ["<pad>", "<unk>", *most_common]
    self.word_index = {word: index for index, word in enumerate(self.vocabulary)}

  def texts_to_sequences(self, texts: Iterable[str]) -> list[list[int]]:
    # Un mot hors du vocabulaire devient 1, <unk>
    return [
      [self.word_index.get(word, 1) for word in self.split(text)] for text in texts
    ]

Transformez les phrases en tenseurs d'indices de mots :

- un `Tokenizer` de 20 000 mots, `tokenizer`, appris sur les seules phrases d'entraînement, et son vocabulaire dans `vocabulary` ;
- une fonction `vectorize(texts)` qui renvoie un tenseur d'une ligne par texte : sa séquence d'indices, tronquée aux 150 premiers, puis complétée par des 0 à la fin jusqu'à la longueur de la plus longue (`pad_sequence`, comme dans les slides) ;
- les phrases d'entraînement transformées dans `X_train`, celles de test dans `X_test`.

Regardez ensuite le résultat : la forme de `X_train`, la première phrase d'entraînement et ses 20 premiers indices, les 10 premiers mots du vocabulaire. Quelle proportion des phrases d'entraînement dépasse 150 mots ? Quelle proportion des mots des phrases de test est hors du vocabulaire ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `Tokenizer(max_tokens=20_000)`, puis `tokenizer.fit_on_texts(X_train_raw)`
- `torch.tensor(sequence[:150])` : les 150 premiers indices d'une séquence, en tenseur
- `pad_sequence(sequences, batch_first=True, padding_value=0)` : une ligne par séquence ; `X_train` a la forme `(51246, 150)`
- `len(text.split())` : le nombre de mots d'une phrase
- `(X_test == 1).sum()` : le nombre de mots inconnus, `(X_test != 0).sum()` celui de mots

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tokenizer = Tokenizer(max_tokens=...)
tokenizer.fit_on_texts(...)
vocabulary = tokenizer.vocabulary


def vectorize(texts: list[str]) -> torch.Tensor:
  sequences = [torch.tensor(...) for sequence in tokenizer.texts_to_sequences(texts)]
  return pad_sequence(sequences, batch_first=True, padding_value=...)


X_train = vectorize(...)
X_test = ...

print(tuple(X_train.shape))
print(X_train_raw[0])
print(X_train[0, :20])
print(" | ".join(vocabulary[:10]))
lengths = np.array([len(text.split()) for text in X_train_raw])
print(f"Phrases de plus de 150 mots : {(lengths > ...).mean():.1%}")
print(f"Mots inconnus en test : {(X_test == ...).sum() / (X_test != ...).sum():.1%}")
```

### Solution

In [ ]:
tokenizer = Tokenizer(max_tokens=20_000)  # les 20 000 mots les plus fréquents
tokenizer.fit_on_texts(X_train_raw)  # vocabulaire de l'entraînement seul
vocabulary = tokenizer.vocabulary


def vectorize(texts: list[str]) -> torch.Tensor:
  # 150 indices au plus par phrase
  sequences = [
    torch.tensor(sequence[:150]) for sequence in tokenizer.texts_to_sequences(texts)
  ]
  # Des 0 à la fin, jusqu'à la longueur de la plus longue séquence
  return pad_sequence(sequences, batch_first=True, padding_value=0)


X_train = vectorize(X_train_raw)
X_test = vectorize(X_test_raw)

print(tuple(X_train.shape))
print(X_train_raw[0])
print(X_train[0, :20])
print(" | ".join(vocabulary[:10]))
lengths = np.array([len(text.split()) for text in X_train_raw])
print(f"Phrases de plus de 150 mots : {(lengths > 150).mean():.1%}")
print(f"Mots inconnus en test : {(X_test == 1).sum() / (X_test != 0).sum():.1%}")

# 0,3 % des phrases dépassent 150 mots : la troncature ne perd presque rien. Les 20 000
# mots gardés, sur les 31 785 des phrases d'entraînement, couvrent 98 % des mots de test :
# 1,8 % seulement sont inconnus (des mots rares, des noms que spaCy n'a pas reconnus).

## 2. Un classifieur récurrent qui ignore le *padding*

Le module Lightning `TextClassifier` de la cellule suivante enveloppe un modèle PyTorch, `model`, qui donne un score (un logit) à chacun des 9 auteurs : il l'entraîne avec la perte d'entropie croisée et l'optimiseur Adam, et mesure sa justesse (*accuracy*) en entraînement, en validation et en test.

In [ ]:
class TextClassifier(lightning.LightningModule):
  """Lightning wrapper: predict the author of a padded sequence of words."""

  def __init__(
    self,
    model: nn.Module,
    learning_rate: float = 1e-3,
    num_classes: int = len(label_encoder.classes_),
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une phrase d'exemple, pour les formes des tenseurs du résumé du modèle :
    # 20 mots inconnus (des 1), car une ligne de 0 serait vide
    self.example_input_array = torch.ones(1, 20, dtype=torch.long)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=num_classes
        )
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    return self.model(sequences)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    sequences, authors = batch
    logits = self(sequences)
    loss = nn.functional.cross_entropy(logits, authors)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, authors)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

### Le modèle récurrent

Écrivez la classe `TextGRU`, un `nn.Module` qui donne leurs scores aux 9 auteurs pour un batch de séquences d'indices, de forme `(batch, longueur)` :

- la couche `embedding` si elle est donnée, sinon une nouvelle couche `nn.Embedding` : un vecteur de 64 nombres pour chacun des mots de `vocabulary`, celui du *padding* (indice 0) restant nul ;
- un *dropout* de taux `dropout` sur les embeddings ;
- une couche `nn.GRU` de `hidden_size` unités et `num_layers` couches, bidirectionnelle si `bidirectional`, avec le même `dropout` entre ses couches quand elle en a plusieurs ;
- une couche `nn.Linear` qui donne les 9 scores à partir des derniers états cachés de la dernière couche du GRU, ceux des deux sens de lecture s'il est bidirectionnel.

Le GRU ne doit pas lire le *padding* (slide « Ignorer le padding dans le RNN ») : une phrase compte 17 mots en médiane, suivis de plus de 130 zéros de *padding* dans `X_train`. Sans précaution, son dernier état caché serait celui du dernier zéro, et la direction arrière d'un GRU bidirectionnel commencerait sa lecture par eux. Passez-lui les séquences empaquetées par `pack_padded_sequence`, avec leurs longueurs réelles : le nombre d'indices non nuls de chaque ligne.

Les parties suivantes créent les modèles à comparer avec les paramètres de `TextGRU`, par exemple `TextGRU(bidirectional=False)`.

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html
- https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

In [ ]:
class TextGRU(nn.Module):
  def __init__(
    self,
    embedding: nn.Embedding | None = None,
    hidden_size: int = 64,
    num_layers: int = 1,
    bidirectional: bool = True,
    dropout: float = 0.0,
  ) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    ...

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Embedding(len(vocabulary), 64, padding_idx=0)` : `padding_idx` garde le vecteur du *padding* nul
- `nn.GRU(embedding.embedding_dim, hidden_size, num_layers=num_layers, bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0, batch_first=True)` : PyTorch avertit d'un `dropout` sans seconde couche, d'où le `0.0`
- `(sequences != 0).sum(dim=1).cpu()` : la longueur réelle de chaque séquence, que `pack_padded_sequence` veut sur le CPU
- `pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)`, comme dans les slides
- `_, h_n = self.gru(packed)` : `h_n` a la forme `(num_layers * 2, batch, hidden_size)` pour un GRU bidirectionnel, et `h_n[-2]`, `h_n[-1]` sont les derniers états des sens avant et arrière de la dernière couche ; `torch.cat([h_n[-2], h_n[-1]], dim=-1)` les met bout à bout
- `nn.Linear(hidden_size * 2, 9)` pour un GRU bidirectionnel, `nn.Linear(hidden_size, 9)` sinon
- `TextGRU()(X_train[:4]).shape` vaut `torch.Size([4, 9])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class TextGRU(nn.Module):
  def __init__(
    self,
    embedding: nn.Embedding | None = None,
    hidden_size: int = 64,
    num_layers: int = 1,
    bidirectional: bool = True,
    dropout: float = 0.0,
  ) -> None:
    super().__init__()
    if embedding is None:
      embedding = nn.Embedding(..., ..., padding_idx=...)
    self.embedding = embedding
    self.dropout = nn.Dropout(...)
    self.gru = nn.GRU(
      embedding.embedding_dim,
      ...,
      num_layers=...,
      bidirectional=...,
      dropout=dropout if num_layers > 1 else 0.0,
      batch_first=True,
    )
    self.head = nn.Linear(..., len(label_encoder.classes_))

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    lengths = (sequences != ...).sum(dim=1).cpu()
    packed = pack_padded_sequence(..., lengths, batch_first=True, enforce_sorted=False)
    _, h_n = self.gru(...)
    if self.gru.bidirectional:
      summary = torch.cat([..., ...], dim=-1)
    else:
      summary = ...
    return self.head(summary)


print(TextGRU()(X_train[:4]).shape)
```

#### Solution

In [ ]:
class TextGRU(nn.Module):
  """Embed the words, summarize them with a GRU, score each author."""

  def __init__(
    self,
    embedding: nn.Embedding | None = None,
    hidden_size: int = 64,
    num_layers: int = 1,
    bidirectional: bool = True,
    dropout: float = 0.0,
  ) -> None:
    super().__init__()
    if embedding is None:
      # Un vecteur de 64 nombres par mot, nul pour le padding
      embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
    self.embedding = embedding
    self.dropout = nn.Dropout(dropout)  # sur les embeddings
    self.gru = nn.GRU(
      embedding.embedding_dim,
      hidden_size,
      num_layers=num_layers,
      bidirectional=bidirectional,
      dropout=dropout if num_layers > 1 else 0.0,  # entre les couches
      batch_first=True,
    )
    directions = 2 if bidirectional else 1
    self.head = nn.Linear(hidden_size * directions, len(label_encoder.classes_))

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    # Longueur réelle de chaque séquence : le GRU saute le padding, et ses
    # derniers états cachés sont ceux des derniers vrais mots, dans les deux sens
    lengths = (sequences != 0).sum(dim=1).cpu()
    packed = pack_padded_sequence(
      embedded, lengths, batch_first=True, enforce_sorted=False
    )
    _, h_n = self.gru(packed)
    if self.gru.bidirectional:
      # Derniers états des sens avant et arrière de la dernière couche
      summary = torch.cat([h_n[-2], h_n[-1]], dim=-1)
    else:
      summary = h_n[-1]
    return self.head(summary)


print(TextGRU()(X_train[:4]).shape)

## 3. Entraîner un GRU bidirectionnel

La fonction `train` de la cellule suivante entraîne un modèle avec Lightning, puis l'évalue :

- le modèle est enveloppé dans un `TextClassifier` ; la norme du gradient est écrêtée à 1 (`gradient_clip_val`), contre l'explosion du gradient, fréquente dans les RNN ;
- des batchs de 256 phrases ; 20 % des phrases d'entraînement en validation, les mêmes à chaque appel ;
- 20 epochs au plus : `EarlyStopping` arrête l'entraînement après 2 epochs sans baisse de la perte de validation, et `ModelCheckpoint` garde les poids de la meilleure, ceux qu'ont le modèle évalué en test et le modèle `model` après l'appel.

Elle affiche le résumé du modèle et ses courbes de justesse, et renvoie sa justesse de test. La cellule d'après l'applique à un GRU bidirectionnel de 64 unités par sens de lecture.

In [ ]:
def train(model: nn.Module, name: str, max_epochs: int = 20) -> float:
  classifier = TextClassifier(model)
  print(ModelSummary(classifier, max_depth=-1))
  # Les mêmes 20 % des phrases d'entraînement en validation à chaque appel
  train_dataset, val_dataset = random_split(
    TensorDataset(X_train, y_train),
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(0),
  )
  # Garde les poids de l'epoch de plus faible perte de validation
  checkpoint = ModelCheckpoint(monitor="val_loss")
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    gradient_clip_val=1.0,  # écrêtage de la norme du gradient
    callbacks=[
      # La barre de progression de tqdm, plutôt que celle de rich que Lightning
      # choisit quand rich est installé (Colab) : elle a figé des entraînements
      TQDMProgressBar(),
      EarlyStopping(monitor="val_loss", patience=2),
      checkpoint,
    ],
    logger=CSVLogger("logs", name=name),
    enable_model_summary=False,  # résumé déjà affiché
  )
  trainer.fit(
    classifier,
    DataLoader(train_dataset, batch_size=256, shuffle=True),
    DataLoader(val_dataset, batch_size=1024),
  )
  # Test avec les poids de la meilleure epoch, rechargés dans model
  test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)
  (result,) = trainer.test(classifier, test_loader, ckpt_path="best", verbose=False)
  accuracy = result["test_accuracy"]

  # Justesses de chaque epoch, écrites par le CSVLogger sur des lignes
  # différentes pour l'entraînement et la validation : on les regroupe
  metrics = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_accuracy", "val_accuracy"])
  )
  best = metrics["val_loss"].idxmin()
  print(
    f"Meilleure epoch : {best + 1} ; justesse d'entraînement "
    f"{metrics['train_accuracy'][best]:.3f}, "
    f"de validation {metrics['val_accuracy'][best]:.3f}, de test {accuracy:.3f}"
  )
  plt.plot(metrics.index + 1, metrics["train_accuracy"], label="entraînement")
  plt.plot(metrics.index + 1, metrics["val_accuracy"], label="validation")
  plt.axvline(best + 1, color="gray", linestyle="--", label="meilleure epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Justesse")
  plt.legend()
  plt.show()
  return accuracy

In [ ]:
bigru = TextGRU()
bigru_accuracy = train(bigru, "bigru")

### Premières conclusions

Au vu des courbes et des justesses, que dire de la qualité de cet apprentissage ? Comparez la justesse de test au hasard (une chance sur 9) et à un modèle qui prédirait toujours l'auteur le plus fréquent, Melville (18 % des phrases).

Votre réponse ici

#### Solution

Le modèle apprend : 55 % de justesse en test, cinq fois le hasard (11 %) et trois fois la classe majoritaire (18 %).

Mais il surapprend : dès la 3e epoch, sa justesse d'entraînement dépasse celle de validation, et l'écart se creuse, jusqu'à 71 % contre 56 % à la meilleure epoch, la 8e. La perte de validation ne baisse presque plus après la 5e epoch, puis remonte : `EarlyStopping` arrête l'entraînement deux epochs plus tard, et `train` évalue les poids de la meilleure.

Le modèle a 1,3 million de paramètres, dont 1,28 million dans la table d'embeddings, pour 41 000 phrases d'entraînement : il retient les mots des phrases d'entraînement plutôt que le style de chaque auteur.

## 4. GRU simple ou bidirectionnel ?

Entraînez avec `train` un GRU simple de 64 unités, qui ne lit les phrases que dans un sens, et comparez-le au GRU bidirectionnel de la partie 3 :

- la justesse de test ;
- le nombre de paramètres de la couche récurrente, et du modèle entier ;
- la durée d'une epoch, que la barre de progression de Lightning affiche pendant l'entraînement.

Le second sens de lecture vaut-il son coût ici ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `TextGRU(bidirectional=False)` : le même modèle, avec un GRU simple
- le résumé qu'affiche `train` donne le nombre de paramètres de chaque couche ; `sum(p.numel() for p in model.parameters())` celui du modèle entier
- `bigru_accuracy` contient la justesse de test du GRU bidirectionnel

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
gru = TextGRU(...)
gru_accuracy = train(..., "gru")
for model, accuracy in ((gru, ...), (bigru, bigru_accuracy)):
  parameters = sum(p.numel() for p in model.parameters())
  print(f"{model.gru} : justesse de test {accuracy:.3f}, {parameters} paramètres")
```

### Solution

In [ ]:
gru = TextGRU(bidirectional=False)
gru_accuracy = train(gru, "gru")
for model, accuracy in ((gru, gru_accuracy), (bigru, bigru_accuracy)):
  parameters = sum(p.numel() for p in model.parameters())
  print(f"{model.gru} : justesse de test {accuracy:.3f}, {parameters} paramètres")

Le GRU simple fait un peu moins bien : 54 % de justesse en test, contre 55 % pour le GRU bidirectionnel. Ce point d'écart est de l'ordre de ce qui change d'une exécution à l'autre.

Le second sens de lecture double les paramètres de la couche récurrente (24 960 → 49 920), mais ce n'est que 2 % du modèle, dont l'embedding fait l'essentiel (1,28 million) ; une epoch dure à peine plus longtemps. Le gain est modeste, mais il ne coûte presque rien : l'état final d'un GRU simple résume surtout la fin de la phrase, et la direction arrière y ajoute un résumé de son début.

## 5. Des embeddings GloVe pré-entraînés et figés

Les embeddings des modèles précédents partent de vecteurs aléatoires et sont appris sur nos 41 000 phrases d'entraînement. Les vecteurs [GloVe](https://nlp.stanford.edu/projects/glove/), eux, ont été appris sur 6 milliards de mots (Wikipédia et Gigaword) : des mots de sens proche y ont des vecteurs proches.

Construisez une couche `glove_embedding` qui donne à chaque mot de `vocabulary` son vecteur GloVe de 300 dimensions, sans le modifier pendant l'entraînement :

- lisez le fichier `glove.6B.300d.txt` dans un dictionnaire `glove`, mot → vecteur ; chaque ligne est un mot suivi de ses 300 coordonnées, séparés par des espaces, et seuls les mots de `vocabulary` nous intéressent ;
- remplissez `embedding_matrix`, de forme `(len(vocabulary), 300)` : sa ligne `i` est le vecteur du mot `vocabulary[i]`, ou reste nulle si GloVe ne connaît pas ce mot ;
- créez `glove_embedding`, une couche `nn.Embedding` initialisée avec cette matrice (`from_pretrained`), figée, et dont le vecteur du *padding* reste nul.

Combien des 20 000 mots du vocabulaire GloVe connaît-il ? Lesquels manquent ?

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html#torch.nn.Embedding.from_pretrained

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `word, *coordinates = line.rstrip().split(" ")` sépare le mot de ses coordonnées ; `np.array(coordinates, dtype="float32")` en fait un vecteur
- `set(vocabulary)` : un test `word in ...` rapide sur les 400 000 lignes du fichier
- `nn.Embedding.from_pretrained(torch.from_numpy(embedding_matrix), freeze=True, padding_idx=0)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in ...:
      glove[word] = np.array(coordinates, dtype="float32")

embedding_matrix = np.zeros((len(vocabulary), 300), dtype="float32")
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = ...
missing = [word for word in vocabulary if word not in ...]
print(f"{len(glove)} mots sur {len(vocabulary)} dans GloVe")
print("Absents :", ", ".join(missing[:30]))

glove_embedding = nn.Embedding.from_pretrained(
  torch.from_numpy(...), freeze=..., padding_idx=...
)
```

### Solution

In [ ]:
# Seuls les vecteurs des mots du vocabulaire sont gardés
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in words:
      glove[word] = np.array(coordinates, dtype="float32")

embedding_matrix = np.zeros(
  (len(vocabulary), 300), dtype="float32"
)  # 0 si le mot est absent
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = glove[word]
missing = [word for word in vocabulary if word not in glove]
print(f"{len(glove)} mots sur {len(vocabulary)} dans GloVe")
print("Absents :", ", ".join(missing[:30]))

glove_embedding = nn.Embedding.from_pretrained(
  torch.from_numpy(embedding_matrix),
  freeze=True,  # table figée
  padding_idx=0,
)

# 19 113 des 20 000 mots du vocabulaire sont dans GloVe. Absents : le padding et <unk>
# (indices 0 et 1), les types d'entités (ppperson, dddate…), des noms que spaCy n'a pas
# reconnus (holgrave, pyncheon), le vocabulaire de la chasse à la baleine (harpooneer,
# whalemen) et des graphies anciennes (d'ye, oftener).

### Entraîner avec GloVe

Entraînez le même GRU bidirectionnel que dans la partie 3, avec `glove_embedding` pour couche d'embedding, et comparez-le au premier :

- l'écart entre la justesse d'entraînement et celle de validation ;
- la justesse de test ;
- le nombre de paramètres entraînables, que le résumé du modèle donne.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`TextGRU` prend la couche d'embedding par son premier paramètre, `embedding`. `bigru_accuracy` contient la justesse de test du premier modèle.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
glove_bigru = TextGRU(...)
glove_accuracy = train(..., "glove_bigru")
print(f"Embeddings appris : {bigru_accuracy:.3f}, GloVe figé : {glove_accuracy:.3f}")
```

#### Solution

In [ ]:
glove_bigru = TextGRU(glove_embedding)
glove_accuracy = train(glove_bigru, "glove_bigru")
print(f"Embeddings appris : {bigru_accuracy:.3f}, GloVe figé : {glove_accuracy:.3f}")

Avec GloVe figé, le modèle n'a plus que 141 705 paramètres entraînables, contre 1,3 million : les 6 millions de nombres de la table GloVe ne bougent pas.

Il généralise mieux : à la meilleure epoch, la 9e, l'écart entre la justesse d'entraînement et celle de validation n'est plus que de 9 points (70 % contre 61 %), au lieu de 15. Sa perte de validation descend plus bas, et la justesse de test passe de 55 % à 59 %. Les vecteurs GloVe apportent ce que nos 41 000 phrases ne peuvent pas apprendre seules : des mots de sens proche y ont des vecteurs proches, même quand ils sont rares dans le corpus.

L'écart finit pourtant par se creuser, à 73 % contre 60 % deux epochs plus tard : la couche récurrente, elle, apprend toujours par cœur. C'est le rôle du *dropout* du modèle suivant.

## 6. Un modèle plus profond

Un dernier modèle, donné : deux couches de GRU bidirectionnel empilées (`num_layers=2`) sur les embeddings GloVe. La seconde lit la séquence des états cachés de la première, mot par mot.

`dropout=0.5` met à zéro, à chaque pas d'entraînement, la moitié des valeurs d'entrée de chaque couche récurrente (les embeddings, puis les états de la première couche), contre le surapprentissage.

In [ ]:
deep = TextGRU(glove_embedding, num_layers=2, dropout=0.5)
deep_accuracy = train(deep, "deep_bigru")
print(
  f"GloVe et un GRU bidirectionnel : {glove_accuracy:.3f}, deux : {deep_accuracy:.3f}"
)

## 7. Les auteurs que le modèle confond

La cellule suivante affiche la matrice de confusion du modèle profond sur les phrases de test : la ligne d'un auteur donne la répartition de ses phrases entre les auteurs prédits.

In [ ]:
deep.eval()  # sans dropout
device = next(deep.parameters()).device
with torch.no_grad():
  y_pred = torch.cat(
    [deep(batch.to(device)).argmax(dim=1).cpu() for batch in X_test.split(1024)]
  )
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=label_encoder.classes_,
  normalize="true",  # proportions de chaque ligne
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Auteur prédit")
ax.set_ylabel("Auteur réel")
plt.show()

### Lecture de la matrice

Quels auteurs le modèle reconnaît-il le mieux ? Lesquels confond-il, et qu'ont-ils en commun ? Les œuvres du corpus sont listées dans le [README du jeu de données](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Votre réponse ici

#### Solution

Dans notre exécution :

- Melville, Shaw et Wilde sont les mieux reconnus (65 % de leurs phrases), puis Cooper (61 %) : *Moby-Dick* a un vocabulaire à part (la mer, la chasse à la baleine), et les pièces de Shaw et de Wilde sont faites de répliques courtes, qui ne ressemblent pas à de la prose.
- Mais Shaw et Wilde se confondent entre eux (14 % et 18 % de leurs phrases) : deux auteurs irlandais de pièces de théâtre, de la même époque.
- Anne Brontë est prise pour sa sœur Charlotte dans 20 % des cas : deux romancières anglaises de la même époque, aux romans racontés à la première personne, par une gouvernante dans *Agnes Grey* comme dans *Jane Eyre*. Charlotte, presque deux fois plus représentée, attire les phrases ambiguës de tous les auteurs : sa colonne est la plus chargée après la diagonale.
- Emerson et Thoreau aussi (18 % et 11 %) : deux essayistes américains du transcendantalisme, voisins et amis.

Une phrase isolée est un indice faible : beaucoup sont courtes et banales (« Yes . »), et aucun modèle ne peut les attribuer. Le modèle profond (60 % en test) ne fait guère mieux que le GRU bidirectionnel sur GloVe (59 %), et une régression logistique sur une représentation TF-IDF des phrases atteint environ 61 % sur ce même découpage : sur des phrases courtes, les mots employés comptent plus que leur ordre.